In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("Api key is set")


Api key is set


In [3]:
# =========================================================
# RAG PIPELINE USING LANGCHAIN + CHROMADB + GROQ
# =========================================================

# =========================================================
# REQUIRED INSTALLATIONS
# =========================================================

# pip install langchain
# pip install langchain-community
# pip install langchain-text-splitters
# pip install chromadb
# pip install sentence-transformers
# pip install langchain-groq


# =========================================================
# STEP 1 — LOAD THE DOCUMENT
# =========================================================

# TextLoader reads the text file and converts it
# into LangChain Document objects.

from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    "company_policy.txt",
    encoding="utf-8"
)

# Load document
documents = loader.load()

# Print metadata of first document
print(documents[0].metadata)


# =========================================================
# STEP 2 — SPLIT DOCUMENT INTO SMALL CHUNKS
# =========================================================

# Large documents cannot be directly stored efficiently.
# We split the document into smaller chunks.

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(

    # Maximum size of each chunk
    chunk_size=1000,

    # Overlap between chunks to preserve context
    chunk_overlap=200
)

# Create chunks from documents
chunks = splitter.split_documents(documents)

# Print total number of chunks
print(f"Total Chunks: {len(chunks)}")


# =========================================================
# STEP 3 — CREATE EMBEDDING MODEL
# =========================================================

# Embeddings convert text into numerical vectors.
# Similar meaning → similar vectors.

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(

    # Lightweight and popular embedding model
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# =========================================================
# STEP 4 — STORE CHUNKS IN CHROMADB
# =========================================================

# ChromaDB stores vector embeddings for similarity search.

from langchain_community.vectorstores import Chroma

vector_store = Chroma.from_documents(

    # Document chunks
    documents=chunks,

    # Embedding model
    embedding=embedding_model,

    # Folder where database is stored
    persist_directory="my_chroma_db"
)

print("ChromaDB Created Successfully")


# =========================================================
# STEP 5 — LOAD VECTOR DATABASE
# =========================================================

# Reload saved Chroma database from disk.

vector_store = Chroma(

    # Existing database folder
    persist_directory="my_chroma_db",

    # Embedding model used during indexing
    embedding_function=embedding_model
)


# =========================================================
# STEP 6 — CREATE RETRIEVER
# =========================================================

# Retriever searches for most relevant chunks.

retriever = vector_store.as_retriever(

    # Similarity search
    search_type="similarity",

    # Return top 2 matching chunks
    search_kwargs={"k": 2}
)


# =========================================================
# STEP 7 — USER QUESTION
# =========================================================

# User query

user_question = "What is a policy?"


# =========================================================
# STEP 8 — RETRIEVE RELEVANT CHUNKS
# =========================================================

# Find most relevant chunks from vector database.

relevant_chunks = retriever.invoke(user_question)

print("\nRetrieved Chunks:\n")

# Print retrieved chunks
for i, chunk in enumerate(relevant_chunks):

    print(f"\nChunk {i+1}:\n")

    print(chunk.page_content)

    print("\n----------------------\n")


# =========================================================
# STEP 9 — BUILD CONTEXT
# =========================================================

# Combine retrieved chunks into one large context.

context = "\n\n".join(
    [chunk.page_content for chunk in relevant_chunks]
)


# =========================================================
# STEP 10 — INITIALIZE LLM
# =========================================================

# Groq LLM model for response generation.

from langchain_groq import ChatGroq

from langchain_core.messages import (
    HumanMessage,
    SystemMessage
)

model = ChatGroq(

    # Fast Groq model
    model="llama-3.1-8b-instant",

    # Deterministic output
    temperature=0.0
)


# =========================================================
# STEP 11 — CREATE PROMPT
# =========================================================

# System prompt controls model behavior.
# Human prompt contains context + question.

messages = [

    SystemMessage(
        content="""
Answer ONLY using the provided context.

If the answer is not available in the context,
reply with:

I don't know
"""
    ),

    HumanMessage(
        content=f"""
Context:
{context}

Question:
{user_question}
"""
    )
]


# =========================================================
# STEP 12 — GENERATE RESPONSE
# =========================================================

# Send prompt to LLM.

response = model.invoke(messages)


# =========================================================
# STEP 13 — PRINT FINAL ANSWER
# =========================================================

print("\nFinal Answer:\n")

print(response.content)

{'source': 'company_policy.txt'}
Total Chunks: 4


C:\Users\Star\AppData\Local\Temp\ipykernel_14024\3239061530.py:72: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB Created Successfully

Retrieved Chunks:


Chunk 1:

Policy is a deliberate system of guidelines to guide decisions and achieve rational outcomes. A policy is a statement of intent and is implemented as a procedure or protocol. Policies are generally adopted by a governance body within an organization. Policies can assist in both subjective and objective decision making. Policies used in subjective decision-making usually assist senior management with decisions that must be based on the relative merits of a number of factors, and as a result, are

----------------------


Chunk 2:

Policy is a deliberate system of guidelines to guide decisions and achieve rational outcomes. A policy is a statement of intent and is implemented as a procedure or protocol. Policies are generally adopted by a governance body within an organization. Policies can assist in both subjective and objective decision making. Policies used in subjective decision-making usually assist senior management with 

C:\Users\Star\AppData\Local\Temp\ipykernel_14024\3239061530.py:108: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(



Final Answer:

A policy is a deliberate system of guidelines to guide decisions and achieve rational outcomes. A policy is a statement of intent and is implemented as a procedure or protocol.


#### **cosine similarity.**

In [4]:
# You can manually check similarity like this
import numpy as np

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

e1 = embedding_model.embed_query("What is the refund policy?")
e2 = embedding_model.embed_query("How do I get my money back?")
e3 = embedding_model.embed_query("What is the weather today?")

print(cosine_similarity(e1, e2))  # ~0.85 → very similar ✅
print(cosine_similarity(e1, e3))  # ~0.15 → not similar  ❌

0.5217813407161381
0.012632300832265513
